# Quarantine Review and Apply

Review quarantined records, apply fixes, and update status for reprocessing.

**Purpose**: Enable manual or automated review and correction of quarantined records

**Workflow**:
1. Query quarantined records by criteria (source, rule, severity)
2. Display records for review
3. Apply fixes (data correction, rule override, manual resolution)
4. Update reprocess_status to RESOLVED for reprocessing

**Status Transitions**:
- PENDING → IN_PROGRESS (during review)
- IN_PROGRESS → RESOLVED (fixed and ready to reprocess)
- IN_PROGRESS → DISCARDED (unfixable, permanent exclusion)

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, current_timestamp, lit, from_json, get_json_object
)
from pyspark.sql.types import StructType
import json

# Configuration
QUARANTINE_TABLE = "workspace.quarantine.quarantine_jobs"

# Status constants
STATUS_PENDING = "PENDING"
STATUS_IN_PROGRESS = "IN_PROGRESS"
STATUS_RESOLVED = "RESOLVED"
STATUS_DISCARDED = "DISCARDED"

In [0]:
def get_quarantine_records(
    source_name: str = None,
    failure_stage: str = None,
    failed_rule: str = None,
    severity: str = None,
    reprocess_status: str = STATUS_PENDING,
    limit: int = 100
) -> DataFrame:
    """
    Query quarantine table with filters.
    
    Args:
        source_name: Filter by source system
        failure_stage: Filter by stage (BRONZE, SILVER, SEMANTIC, WAREHOUSE)
        failed_rule: Filter by specific rule
        severity: Filter by severity (ERROR, WARNING)
        reprocess_status: Filter by status (default PENDING)
        limit: Max records to return
        
    Returns:
        DataFrame of quarantined records
    """
    df = spark.table(QUARANTINE_TABLE)
    
    # Apply filters
    if source_name:
        df = df.filter(col("source_name") == source_name)
    if failure_stage:
        df = df.filter(col("failure_stage") == failure_stage)
    if failed_rule:
        df = df.filter(col("failed_rule_name") == failed_rule)
    if severity:
        df = df.filter(col("severity") == severity)
    if reprocess_status:
        df = df.filter(col("reprocess_status") == reprocess_status)
    
    return df.orderBy(col("quarantined_at").desc()).limit(limit)

In [0]:
def display_quarantine_summary():
    """
    Display summary statistics of quarantined records.
    """
    df = spark.table(QUARANTINE_TABLE)
    
    print("=" * 80)
    print("QUARANTINE SUMMARY")
    print("=" * 80)
    
    # Overall counts by status
    print("\n📊 Records by Status:")
    df.groupBy("reprocess_status").count().orderBy("count", ascending=False).show()
    
    # Pending by failure stage
    print("\n🔍 Pending Records by Stage:")
    df.filter(col("reprocess_status") == STATUS_PENDING) \
        .groupBy("failure_stage").count() \
        .orderBy("count", ascending=False).show()
    
    # Top failed rules
    print("\n⚠️  Top Failed Rules (Pending):")
    df.filter(col("reprocess_status") == STATUS_PENDING) \
        .groupBy("failed_rule_name", "severity").count() \
        .orderBy("count", ascending=False).limit(10).show()
    
    # Records by source
    print("\n📦 Pending Records by Source:")
    df.filter(col("reprocess_status") == STATUS_PENDING) \
        .groupBy("source_name").count() \
        .orderBy("count", ascending=False).show()

# Run summary
display_quarantine_summary()

In [0]:
# Example: Review pending records for a specific rule
print("\n" + "=" * 80)
print("REVIEWING SPECIFIC QUARANTINE RECORDS")
print("=" * 80)

# Get records for review (adjust filters as needed)
review_df = get_quarantine_records(
    source_name="job_postings_linkedin",  # Change as needed
    failed_rule="salary_validation",       # Change as needed
    reprocess_status=STATUS_PENDING,
    limit=10
)

print(f"\n📋 Found {review_df.count()} records for review\n")

# Display key fields
review_df.select(
    "quarantine_id",
    "source_name",
    "failed_rule_name",
    "severity",
    "quarantined_at",
    "reprocess_status"
).show(truncate=False)

# Display payload details for first few records
print("\n💾 Sample Payload Data:")
for row in review_df.limit(3).collect():
    print(f"\nQuarantine ID: {row['quarantine_id']}")
    print(f"Rule: {row['failed_rule_name']}")
    try:
        payload = json.loads(row['payload_json'])
        print(f"Payload: {json.dumps(payload, indent=2)}")
    except:
        print(f"Payload (raw): {row['payload_json'][:200]}...")
    print("-" * 60)

In [0]:
def update_quarantine_status(
    quarantine_ids: list,
    new_status: str,
    reprocess_batch_id: str = None
) -> int:
    """
    Update reprocess_status for specific quarantine records.
    
    Args:
        quarantine_ids: List of quarantine_id values to update
        new_status: New status (IN_PROGRESS, RESOLVED, DISCARDED)
        reprocess_batch_id: Batch ID for reprocessing (if RESOLVED)
        
    Returns:
        Count of records updated
    """
    if not quarantine_ids:
        print("No quarantine IDs provided")
        return 0
    
    # Build update statement
    ids_str = "', '".join(quarantine_ids)
    
    update_sql = f"""
    UPDATE {QUARANTINE_TABLE}
    SET reprocess_status = '{new_status}',
        resolved_at = CASE WHEN '{new_status}' IN ('RESOLVED', 'DISCARDED') 
                          THEN current_timestamp() 
                          ELSE resolved_at END,
        reprocess_batch_id = CASE WHEN '{new_status}' = 'RESOLVED' 
                                  THEN '{reprocess_batch_id}' 
                                  ELSE reprocess_batch_id END
    WHERE quarantine_id IN ('{ids_str}')
    """
    
    spark.sql(update_sql)
    
    count = len(quarantine_ids)
    print(f"✓ Updated {count} records to status: {new_status}")
    
    return count


def mark_resolved(quarantine_ids: list, reprocess_batch_id: str = "REPROCESS_TBD"):
    """Mark records as RESOLVED and ready for reprocessing."""
    return update_quarantine_status(quarantine_ids, STATUS_RESOLVED, reprocess_batch_id)


def mark_discarded(quarantine_ids: list):
    """Mark records as DISCARDED (unfixable, exclude permanently)."""
    return update_quarantine_status(quarantine_ids, STATUS_DISCARDED)


def mark_in_progress(quarantine_ids: list):
    """Mark records as IN_PROGRESS (under review)."""
    return update_quarantine_status(quarantine_ids, STATUS_IN_PROGRESS)

In [0]:
# Example workflow: Mark records as resolved after manual fix

# Step 1: Get specific records to fix
records_to_fix = get_quarantine_records(
    failed_rule="salary_validation",
    reprocess_status=STATUS_PENDING,
    limit=5
)

print(f"\n🔧 Processing {records_to_fix.count()} records for resolution\n")

# Step 2: Extract quarantine IDs
quarantine_ids = [row['quarantine_id'] for row in records_to_fix.collect()]

# Step 3a: Mark as IN_PROGRESS during review (optional)
# mark_in_progress(quarantine_ids)

# Step 3b: After fixing data issues, mark as RESOLVED
# Note: Actual data fixes would be applied to the original source/staging tables
# The quarantine record status indicates it's ready for reprocessing

if quarantine_ids:
    updated = mark_resolved(
        quarantine_ids=quarantine_ids,
        reprocess_batch_id="REPROCESS_20240601_001"
    )
    
    print(f"\n✅ {updated} records marked as RESOLVED and ready for reprocessing")
    print("Next step: Run quarantine_reprocess_batch notebook")
else:
    print("\nℹ️  No records to update")

In [0]:
def discard_by_criteria(
    source_name: str = None,
    failed_rule: str = None,
    severity: str = None,
    older_than_days: int = None
) -> int:
    """
    Bulk discard records matching criteria (e.g., known unfixable issues).
    
    Args:
        source_name: Source system name
        failed_rule: Failed rule name
        severity: Severity level
        older_than_days: Only discard records older than N days
        
    Returns:
        Count of records discarded
    """
    df = spark.table(QUARANTINE_TABLE).filter(col("reprocess_status") == STATUS_PENDING)
    
    if source_name:
        df = df.filter(col("source_name") == source_name)
    if failed_rule:
        df = df.filter(col("failed_rule_name") == failed_rule)
    if severity:
        df = df.filter(col("severity") == severity)
    if older_than_days:
        df = df.filter(col("quarantined_at") < current_timestamp() - expr(f"INTERVAL {older_than_days} DAYS"))
    
    ids_to_discard = [row['quarantine_id'] for row in df.collect()]
    
    if ids_to_discard:
        return mark_discarded(ids_to_discard)
    else:
        print("No matching records found to discard")
        return 0

# Example: Discard old records from deprecated source
# discard_count = discard_by_criteria(
#     source_name="legacy_system_xyz",
#     older_than_days=90
# )
# print(f"Discarded {discard_count} records from deprecated source")